# 11 — Distributional Word Embeddings from Co-occurrence + SVD

**Learning objective.** Build dense semantic vectors from the distributional hypothesis and visualize nearest neighbors.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
from sklearn.decomposition import TruncatedSVD
sentences=[
 'cat pet animal', 'dog pet animal', 'cat likes milk', 'dog likes bone',
 'car vehicle road', 'truck vehicle road', 'car uses fuel', 'truck uses fuel']
toks=[s.split() for s in sentences]
vocab=sorted(set(sum(toks,[]))); idx={w:i for i,w in enumerate(vocab)}
C=np.zeros((len(vocab),len(vocab)))
for sent in toks:
    for i,w in enumerate(sent):
        for j in range(max(0,i-2),min(len(sent),i+3)):
            if i!=j: C[idx[w],idx[sent[j]]]+=1
svd=TruncatedSVD(n_components=3,random_state=42)
E=svd.fit_transform(C)
print('embedding matrix:',E.shape)

embedding matrix: (13, 3)


In [3]:
from sklearn.metrics.pairwise import cosine_similarity
S=cosine_similarity(E)
def neighbors(word,k=3):
    i=idx[word]
    order=np.argsort(-S[i])
    return [(vocab[j],round(float(S[i,j]),3)) for j in order if j!=i][:k]
for w in ['cat','dog','car','truck']:
    print(w, neighbors(w))

cat [('dog', 1.0), ('milk', 0.979), ('animal', 0.979)]
dog [('cat', 1.0), ('milk', 0.979), ('animal', 0.979)]
car [('truck', 1.0), ('vehicle', 0.985), ('uses', 0.982)]
truck [('car', 1.0), ('vehicle', 0.985), ('uses', 0.982)]


In [4]:

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(E[:,0],E[:,1])
for i,w in enumerate(vocab): ax.annotate(w,(E[i,0],E[i,1]),xytext=(4,4),textcoords='offset points')
ax.set_title('Dense word vectors projected to first two SVD dimensions')
ax.set_xlabel('component 1'); ax.set_ylabel('component 2')
plt.tight_layout(); plt.show()


[static visualization generated successfully during execution; rerun in Jupyter/VS Code to display]


---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- State the distributional hypothesis
- Explain how dimensionality reduction creates dense vectors